# PILOT — CE-crossing selector (gold-free) tren Forget-MI

Kiem tra quy tac chon checkpoint KHONG dung GOLD, KHONG dung final test:
> chon epoch DAU TIEN co **forget_ce >= nm_val_ce** (CE tren D_f >= CE tren non-member D_nm_val).

Luong: (Cell 3) chay Forget-MI **non-dual** -> luu 30 checkpoint E0..E29;
(Cell 4) selector doc 30 ckpt, tach test 25/75 theo benh nhan, tinh CE, chon epoch;
(Cell 5) xem ket qua. KHONG doi training/loss Forget-MI.

**Luu y:** non-dual luu 30 x ~450MB = ~13.5GB (trong /working, session-only). Cell 6 don bot sau.


In [ ]:
# Cell 1: setup
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/ce_selector_pilot.py'),'push code truoc + re-import notebook'
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: path discovery (giong baseline)
import glob, os
FORGET_PCT = 3        # 3 | 6 | 10
SEED       = 42
assert FORGET_PCT in (3,6,10)

def find_dataset(*slugs):
    for slug in slugs:
        if os.path.isdir(f'/kaggle/input/{slug}'): return f'/kaggle/input/{slug}'
        hits=glob.glob(f'/kaggle/input/datasets/*/{slug}')
        if hits: return sorted(hits)[0]
    return None
def first_existing(root, rels):
    for r in rels:
        p=os.path.join(root,r)
        if os.path.exists(p): return p
    return None

DATA_ROOT=find_dataset('forget-mi-data')
MODELS_ROOT=find_dataset('forget-mi-models-full','forget-mi-models-v2','forget-mi-models')
assert DATA_ROOT and MODELS_ROOT,'Add forget-mi-data + forget-mi-models-full'
base_hits=glob.glob(os.path.join(MODELS_ROOT,'**','training_original_model','pytorch_model.bin'),recursive=True)
gold_hits=glob.glob(os.path.join(MODELS_ROOT,'**',f'model_retrained_{FORGET_PCT}per','**','pytorch_model.bin'),recursive=True)
BASE_MODEL=os.path.dirname(sorted(base_hits,key=len)[0]) if base_hits else None
GOLD_MODEL=os.path.dirname(sorted(gold_hits,key=len)[0]) if gold_hits else BASE_MODEL
TEXT_DIR=first_existing(DATA_ROOT,['data/metadata','metadata'])
IMG_DIR =first_existing(DATA_ROOT,['data/img_data','img_data'])
FORGET_CSV=f'./data_splits/forget_set_{FORGET_PCT}per.csv'

RUN_ID    =f'forgetmi_pilot_{FORGET_PCT}per_s{SEED}'
OUTPUT_DIR=f'/kaggle/working/pilot_output/{FORGET_PCT}per_s{SEED}'
SEL_DIR   =f'/kaggle/working/checkpoint_selection_{FORGET_PCT}per_s{SEED}'
for n,p in {'base':BASE_MODEL,'text':TEXT_DIR,'img':IMG_DIR,'forget':FORGET_CSV}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'

COMMON_OVR={'forget_set_path':FORGET_CSV,'base_model_path':BASE_MODEL,'bert_pretrained_dir':BASE_MODEL,
            'retrained_model_path':GOLD_MODEL,'text_data_dir':TEXT_DIR,'img_data_dir':IMG_DIR}
print('FORGET_PCT',FORGET_PCT,'SEED',SEED); print('BASE',BASE_MODEL); print('OUT',OUTPUT_DIR)


In [ ]:
# Cell 3: chay Forget-MI NON-DUAL (luu 30 checkpoint E0..E29). ~2.5-3h.
#   evaluate_last_and_best=0 -> luu epoch_<e>/model_state_dict.pth moi epoch (hanh vi GOC).
#   eval_every_epoch=0       -> bo eval-moi-epoch (dung gold) cho nhanh; selector tu tinh CE sau.
#   KHONG doi training/loss.
import os, subprocess, time
ovr=dict(COMMON_OVR); ovr.update({'output_dir':OUTPUT_DIR,
    'results_csv_path':'/kaggle/working/results_pilot_native.csv',
    'evaluate_last_and_best':0, 'eval_every_epoch':0, 'id':RUN_ID})
arg=','.join(f'{k}={v}' for k,v in ovr.items())
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
cmd=['python','training/forgetmi_partial.py','--config','config_baseline_kaggle.yaml',
     '--seed',str(SEED),'--fresh','--override',arg]
print('='*70); print('TRAIN (non-dual, 30 ckpt)',RUN_ID); print('='*70)
t0=time.time()
try:
    subprocess.run(cmd,env=env,check=True); print(f'DONE train ({(time.time()-t0)/3600:.2f}h)')
except subprocess.CalledProcessError as e:
    print(f'FAIL train rc={e.returncode}')


In [ ]:
# Cell 4: chay CE-crossing selector tren 30 checkpoint da luu
import os, glob, subprocess
hits=sorted(glob.glob(f'{OUTPUT_DIR}/**/epoch_0/model_state_dict.pth',recursive=True),key=len)
assert hits, f'Khong thay epoch_0/model_state_dict.pth trong {OUTPUT_DIR} (Cell 3 co chay non-dual chua?)'
CKPT_DIR=os.path.dirname(os.path.dirname(hits[0]))
n_ckpt=len(glob.glob(f'{CKPT_DIR}/epoch_*/model_state_dict.pth'))
print('CKPT_DIR',CKPT_DIR,'| so checkpoint =',n_ckpt)

arg=','.join(f'{k}={v}' for k,v in COMMON_OVR.items())
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
cmd=['python','training/ce_selector_pilot.py','--config','config_baseline_kaggle.yaml',
     '--seed',str(SEED),'--override',arg,'--checkpoints_dir',CKPT_DIR,
     '--out_dir',SEL_DIR,'--max_epochs',str(n_ckpt),'--split_seed','42','--nm_val_ratio','0.25']
subprocess.run(cmd,env=env,check=True)


In [ ]:
# Cell 5: xem ket qua 4 selector
import os, json, pandas as pd
pd.set_option('display.width',180)
traj=os.path.join(SEL_DIR,'forgetmi_selector.csv')
if os.path.exists(traj):
    print('===== TRAJECTORY (gold-free: forget_ce vs nm_val_ce + utility) =====')
    print(pd.read_csv(traj).to_string(index=False))
sj=os.path.join(SEL_DIR,'selected_checkpoints.json')
if os.path.exists(sj):
    d=json.load(open(sj)); res=d['results']
    print(f"\n===== 4 CACH CHON (s4_delta={d['s4_delta']}, s4_match={d.get('s4_match_epochs')}) =====")
    tab=[]
    for k,v in res.items():
        if v.get('epoch') is None: tab.append({'selector':k,'epoch':'NO CROSSING'})
        else: tab.append({'selector':k,'epoch':f"E{v['epoch']}",'Df_AUC':v['Df_AUC'],'Df_F1':v['Df_F1'],
                          'Dt_AUC':v['Dt_AUC'],'Dt_F1':v['Dt_F1'],'MIA':v['MIA']})
    print(pd.DataFrame(tab).to_string(index=False))
    uniq=sorted(set(v['epoch'] for v in res.values() if v.get('epoch') is not None))
    print(f"\n-> {len(uniq)} epoch khac nhau: {['E'+str(e) for e in uniq]}  "
          f"({'DONG THUAN cao' if len(uniq)<=2 else 'phan tan'})")
print('\nOutput:',SEL_DIR)


In [ ]:
# Cell 6 (tuy chon): don bot checkpoint de giam dung luong output
#   Giu epoch_<selected> + epoch_29 (last), xoa cac epoch con lai.
import os, json, glob, shutil
sj=os.path.join(SEL_DIR,'selected_checkpoints.json')
keep=set()
if os.path.exists(sj):
    d=json.load(open(sj)); e=d.get('forgetmi',{}).get('epoch')
    if e is not None: keep.add(int(e))
import glob as _g
alle=[int(os.path.basename(os.path.dirname(p)).split('_')[1])
      for p in _g.glob(f'{OUTPUT_DIR}/**/epoch_*/model_state_dict.pth',recursive=True)]
if alle: keep.add(max(alle))   # last
hits=_g.glob(f'{OUTPUT_DIR}/**/epoch_*/model_state_dict.pth',recursive=True)
removed=0
for p in hits:
    e=int(os.path.basename(os.path.dirname(p)).split('_')[1])
    if e not in keep:
        shutil.rmtree(os.path.dirname(p),ignore_errors=True); removed+=1
print(f'Giu epoch {sorted(keep)}, da xoa {removed} epoch-dir khac.')
